In [1]:
!pip install timm -q


In [2]:
import os
import torch
import timm
import pandas as pd
import torch.nn as nn
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import classification_report, confusion_matrix


In [4]:
CSV_PATH = "/kaggle/input/datasets/djjagatra/updated-dataset/updated_dataset.csv"
IMG_ROOT = "/kaggle/input/datasets/djjagatra/chart-img/charts - Copy"

df = pd.read_csv(CSV_PATH)

def fix_path(p):
    p = str(p).replace("\\", "/")
    ticker = p.split("/")[-2]
    file = os.path.basename(p)
    return f"{IMG_ROOT}/{ticker}/{file}"

df["image_path"] = df["image_path"].apply(fix_path)
df["date"] = pd.to_datetime(df["date"], format="mixed", dayfirst=True)
df = df.sort_values(["date", "ticker"]).reset_index(drop=True)


print(df.head())
print(df["label_id"].value_counts())


                                          image_path ticker       date  \
0  /kaggle/input/datasets/djjagatra/chart-img/cha...   AAPL 2022-02-01   
1  /kaggle/input/datasets/djjagatra/chart-img/cha...   AMZN 2022-02-01   
2  /kaggle/input/datasets/djjagatra/chart-img/cha...  GOOGL 2022-02-01   
3  /kaggle/input/datasets/djjagatra/chart-img/cha...   META 2022-02-01   
4  /kaggle/input/datasets/djjagatra/chart-img/cha...   MSFT 2022-02-01   

     label  label_id    RSI  MACD  MACD_signal  forward_return  trend_score  \
0  Neutral         1  44.24   0.0          0.0         -0.0092      -0.1367   
1  Neutral         1  33.55   0.0          0.0          0.0352       0.4791   
2  Neutral         1  47.91   0.0          0.0         -0.0075      -0.1237   
3     Down         0  45.22   0.0          0.0         -0.3072      -4.1277   
4  Neutral         1  33.94   0.0          0.0         -0.0268      -0.4339   

   split  
0  train  
1  train  
2  train  
3  train  
4  train  
label_id
1    

In [7]:
unique_dates = sorted(df["date"].unique())

train_date_end = unique_dates[int(0.70 * len(unique_dates))]
val_date_end = unique_dates[int(0.90 * len(unique_dates))]

train_df = df[df["date"] < train_date_end].reset_index(drop=True)
val_df = df[(df["date"] >= train_date_end) & (df["date"] < val_date_end)].reset_index(drop=True)
test_df = df[df["date"] >= val_date_end].reset_index(drop=True)

print("Train:", train_df["date"].min(), train_df["date"].max(), len(train_df))
print("Val:", val_df["date"].min(), val_df["date"].max(), len(val_df))
print("Test:", test_df["date"].min(), test_df["date"].max(), len(test_df))

print("Train labels:", train_df["label_id"].value_counts().to_dict())
print("Val labels:", val_df["label_id"].value_counts().to_dict())
print("Test labels:", test_df["label_id"].value_counts().to_dict())



Train: 2022-02-01 00:00:00 2024-10-15 00:00:00 5440
Val: 2024-10-16 00:00:00 2025-07-28 00:00:00 1552
Test: 2025-07-29 00:00:00 2025-12-15 00:00:00 784
Train labels: {1: 2065, 2: 2020, 0: 1355}
Val labels: {2: 617, 1: 542, 0: 393}
Test labels: {1: 339, 2: 259, 0: 186}


In [8]:
class StockDataset(Dataset):
    def __init__(self, dataframe, train=True):
        self.df = dataframe

        if train:
            self.transform = transforms.Compose([
                transforms.Resize((224,224)),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
                transforms.ToTensor(),
                transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((224,224)),
                transforms.ToTensor(),
                transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        image = self.transform(image)

        features = torch.tensor([
            row["RSI"],
            row["MACD"],
            row["trend_score"]
        ], dtype=torch.float32)

        label = torch.tensor(row["label_id"], dtype=torch.long)

        return image, features, label


In [9]:
train_loader = DataLoader(StockDataset(train_df, train=True), batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(StockDataset(val_df, train=False), batch_size=16, shuffle=False, num_workers=2)
test_loader = DataLoader(StockDataset(test_df, train=False), batch_size=16, shuffle=False, num_workers=2)


In [10]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.vit = timm.create_model("vit_small_patch16_224", pretrained=True)
        self.vit.head = nn.Identity()

        self.fc = nn.Sequential(
            nn.Linear(384 + 3, 128),
            nn.ReLU(),
            nn.Dropout(0.6),
            nn.Linear(128, 3)
        )

    def forward(self, img, features):
        img_feat = self.vit(img)
        x = torch.cat((img_feat, features), dim=1)
        return self.fc(x)


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model().to(device)

for param in model.vit.parameters():
    param.requires_grad = False

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW([
    {"params": model.vit.parameters(), "lr": 5e-6},
    {"params": model.fc.parameters(), "lr": 5e-5}
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-6
)

scaler = torch.amp.GradScaler("cuda")

best_val_loss = float("inf")
patience = 3
counter = 0
EPOCHS = 40


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

In [12]:
for epoch in range(EPOCHS):

    if epoch == 8:
        print(" Unfreezing ViT + resetting optimizer")

        for param in model.vit.parameters():
            param.requires_grad = True

        optimizer = torch.optim.AdamW([
            {"params": model.vit.parameters(), "lr": 5e-6},
            {"params": model.fc.parameters(), "lr": 5e-5}
        ], weight_decay=1e-4)

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-6
        )

    print(f"\n Epoch {epoch+1}/{EPOCHS} - Training...")

    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for img, features, labels in tqdm(train_loader, leave=False):
        img = img.to(device)
        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model(img, features)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss /= len(train_loader)
    train_acc = correct / total

    print(" Validating...")

    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for img, features, labels in tqdm(val_loader, leave=False):
            img = img.to(device)
            features = features.to(device)
            labels = labels.to(device)

            with torch.amp.autocast("cuda"):
                outputs = model(img, features)
                loss = criterion(outputs, labels)

            val_loss += loss.item()
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total

    scheduler.step(val_loss)

    print(f"\n Epoch [{epoch+1}/{EPOCHS}]")
    print(f"   Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"   Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print(f"   LR: {optimizer.param_groups[0]['lr']:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), "/kaggle/working/best_model_2.pth")
        print(" Best model saved")
    else:
        counter += 1
        print(f" Patience: {counter}/{patience}")

        if counter >= patience:
            print(" Early stopping triggered")
            break



 Epoch 1/40 - Training...


 Validating...



 Epoch [1/40]
   Train Loss: 1.1868 | Train Acc: 0.3726
   Val   Loss: 1.0806 | Val   Acc: 0.3911
   LR: 0.000005
 Best model saved

 Epoch 2/40 - Training...


 Validating...



 Epoch [2/40]
   Train Loss: 1.0834 | Train Acc: 0.4068
   Val   Loss: 1.0669 | Val   Acc: 0.4175
   LR: 0.000005
 Best model saved

 Epoch 3/40 - Training...


...


 Epoch [18/40]
   Train Loss: 0.7096 | Train Acc: 0.7768
   Val   Loss: 0.7413 | Val   Acc: 0.7332
   LR: 0.000005
 Patience: 1/3

 Epoch 19/40 - Training...


 Validating...



 Epoch [19/40]
   Train Loss: 0.6853 | Train Acc: 0.7947
   Val   Loss: 0.7538 | Val   Acc: 0.7171
   LR: 0.000005
 Patience: 2/3

 Epoch 20/40 - Training...


 Validating...



 Epoch [20/40]
   Train Loss: 0.6663 | Train Acc: 0.8063
   Val   Loss: 0.7867 | Val   Acc: 0.6624
   LR: 0.000003
 Patience: 3/3
 Early stopping triggered


In [13]:
from sklearn.metrics import classification_report, confusion_matrix

model.load_state_dict(torch.load("/kaggle/working/best_model_2.pth"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for img, features, labels in tqdm(test_loader):
        img = img.to(device)
        features = features.to(device)
        labels = labels.to(device)

        with torch.amp.autocast("cuda"):
            outputs = model(img, features)

        preds = outputs.argmax(1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Test Report:")
print(classification_report(all_labels, all_preds, target_names=["Down", "Neutral", "Up"]))

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))


100%|██████████| 49/49 [00:04<00:00, 12.24it/s]

Test Report:
              precision    recall  f1-score   support

        Down       0.92      0.57      0.70       186
     Neutral       0.70      0.67      0.68       339
          Up       0.69      0.92      0.79       259

    accuracy                           0.73       784
   macro avg       0.77      0.72      0.73       784
weighted avg       0.75      0.73      0.72       784

Confusion Matrix:
[[106  77   3]
 [  9 228 102]
 [  0  22 237]]
